# Packshot BYOC Example

This notebook demonstrates the BYOC package flow for **Packshot**: request a short-lived CodeArtifact credential from the Bria Engine, install **`packshot`** from the **`bria-packshot`** repository, then run the local `Packshot` pipeline on product images.

The public Product Packshot docs are useful background for the product-shot workflow and expected outputs: https://docs.bria.ai/product-shot-editing/product-endpoints/product-packshot

**Prerequisites**

- `BRIA_API_TOKEN` in the environment.
- `HF_TOKEN` in the environment with access to `briaai/RMBG-2.0`.
- Python 3.10 or newer.
- Network access to Bria Engine, AWS CodeArtifact, and Hugging Face.
- Runtime support for the underlying `cutout` model package.

## 1. CodeArtifact Token

Request a short-lived credential for the **`bria-packshot`** PyPI repository. The credential is used as the CodeArtifact password with username `aws`.

In [ ]:
import json
import os
from urllib.error import HTTPError
from urllib.parse import urlencode
from urllib.request import Request, urlopen

BRIA_API_TOKEN = os.environ.get("BRIA_API_TOKEN")
if not BRIA_API_TOKEN:
    raise RuntimeError("Set BRIA_API_TOKEN in the environment before running this notebook.")

repository = "bria-packshot"
params = urlencode({"repository": repository})
url = f"https://engine.prod.bria-api.com/v2/auth/access/code_artifact?{params}"
request = Request(
    url,
    headers={
        "api_token": BRIA_API_TOKEN,
        "token": BRIA_API_TOKEN,
        "User-Agent": "BriaPlatform/APIdocs/LLMsAgent",
    },
)

try:
    with urlopen(request, timeout=60) as response:
        payload = json.loads(response.read().decode("utf-8"))
except HTTPError as exc:
    error_body = exc.read().decode("utf-8", errors="replace")
    try:
        error_payload = json.dumps(json.loads(error_body), indent=2)
    except json.JSONDecodeError:
        error_payload = error_body or "<empty response body>"
    raise RuntimeError(
        f"CodeArtifact token request failed for repository {repository!r} "
        f"with HTTP {exc.code}: {exc.reason}\n{error_payload}"
    ) from exc

result = payload.get("result") or {}
CODE_ARTIFACT_PASSWORD = (result.get("authorization_token") or "").strip()
if not CODE_ARTIFACT_PASSWORD:
    raise RuntimeError(f"CodeArtifact response did not include an authorization token: {payload}")

expiration = result.get("expiration")
print("CodeArtifact credential acquired." + (f" Expires: {expiration}" if expiration else ""))

## 2. Install dependencies

In [ ]:
import os
import subprocess
import sys
from pathlib import Path
from urllib.parse import quote

# Pip imports call os.getcwd(). If Jupyter started from a folder that was later removed, inherited cwd breaks pip.
try:
    pip_cwd = str(Path.cwd())
except FileNotFoundError:
    pip_cwd = str(Path.home())
    os.chdir(pip_cwd)

encoded_password = quote(CODE_ARTIFACT_PASSWORD, safe="")
packshot_index = (
    "https://aws:"
    + encoded_password
    + "@bria-300465780738.d.codeartifact.us-east-1.amazonaws.com/pypi/bria-packshot/simple/"
)

pytorch_index = "https://download.pytorch.org/whl/cu126"


def run_pip_install(install_cmd):
    redacted_cmd = " ".join(part.replace(encoded_password, "<redacted>") for part in install_cmd)
    print("Running:", redacted_cmd, flush=True)
    process = subprocess.Popen(
        install_cmd,
        cwd=pip_cwd,
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        bufsize=1,
    )

    assert process.stdout is not None
    for line in process.stdout:
        print(line.replace(encoded_password, "<redacted>"), end="", flush=True)

    returncode = process.wait()
    if returncode != 0:
        raise RuntimeError("pip install failed. See pip output above; authenticated index URLs were redacted.")


run_pip_install(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "--upgrade",
        "--force-reinstall",
        "torch",
        "torchvision",
        "--index-url",
        pytorch_index,
    ]
)

run_pip_install(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "--upgrade",
        "--upgrade-strategy",
        "only-if-needed",
        "packshot",
        "--extra-index-url",
        packshot_index,
    ]
)

## 3. Start The Local Pipeline

The example disables model compilation for a simpler first run. Tune the nested `cutout_config` for your deployment environment.

If the install cell changed core runtime packages, restart the notebook kernel before running this setup cell so Python imports the installed versions cleanly.

In [ ]:
import logging
import os
from pathlib import Path

from IPython.display import display
from packshot.packshot import Packshot
from packshot.schemas import PackshotConfig, PackshotInput

logging.basicConfig(level=logging.INFO)

outputs_dir = Path("outputs")
outputs_dir.mkdir(exist_ok=True)

packshot = Packshot(config=PackshotConfig(cutout_config={"rmbg_config": {"compile_model": False}}))
packshot.setup()
print("Packshot pipeline setup complete.")

## 4. Generate A White-Background Packshot

Pass either a URL or a local image path into `PackshotInput.image`. The result is centered on a `2000x2000` canvas.

In [ ]:
import time

product_image = "https://bria-test-images.s3.us-east-1.amazonaws.com/strawberry.jpg"

task = PackshotInput(
    image=product_image,
    background_color="#FFFFFF",
    preserve_alpha=False,
)

t0 = time.time()
white_result = packshot.execute(task)
white_output_path = outputs_dir / "strawberry_packshot_white.png"
white_result.image.save(white_output_path)

print(f"Generated {white_result.image.size} packshot in {time.time() - t0:.1f}s")
print(f"Saved to {white_output_path}")
display(white_result.image)

## 5. Generate A Transparent Packshot

Use `background_color="transparent"` and `preserve_alpha=True` when the output needs alpha.

In [ ]:
transparent_task = PackshotInput(
    image=product_image,
    background_color="transparent",
    preserve_alpha=True,
)

t0 = time.time()
transparent_result = packshot.execute(transparent_task)
transparent_output_path = outputs_dir / "strawberry_packshot_transparent.png"
transparent_result.image.save(transparent_output_path)

print(f"Generated {transparent_result.image.size} transparent packshot in {time.time() - t0:.1f}s")
print(f"Saved to {transparent_output_path}")
display(transparent_result.image)

## 6. Cleanup

Release resources held by the underlying pipeline when you are done.

In [ ]:
packshot.cleanup()
print("Cleanup done.")